Notebook: 08_failure_modes.ipynb

Purpose: Identify systematic failure modes that impact confidence.

Inputs:
- signal_quality_features.parquet
- delineation_features.parquet
- twave_features.parquet
- measurement_reliability.parquet

Outputs:
- failure_modes.parquet

# 08 — Failure Modes

Detect record-level failure categories using multi-domain artifact evidence.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent / 'src'))

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
frames = {
    'signal_quality': pd.read_parquet(artifacts_dir / 'signal_quality_features.parquet') if (artifacts_dir / 'signal_quality_features.parquet').exists() else pd.DataFrame(),
    'delineation': pd.read_parquet(artifacts_dir / 'delineation_features.parquet') if (artifacts_dir / 'delineation_features.parquet').exists() else pd.DataFrame(),
    'twave': pd.read_parquet(artifacts_dir / 'twave_features.parquet') if (artifacts_dir / 'twave_features.parquet').exists() else pd.DataFrame(),
    'measurement': pd.read_parquet(artifacts_dir / 'measurement_reliability.parquet') if (artifacts_dir / 'measurement_reliability.parquet').exists() else pd.DataFrame(),
}

record_ids = set()
for df in frames.values():
    if 'record_id' in df.columns:
        record_ids.update(df['record_id'].astype(str).unique())

rows = []
for record_id in sorted(record_ids):
    sq = frames['signal_quality']
    de = frames['delineation']
    tw = frames['twave']
    mr = frames['measurement']

    sq_rec = sq[sq['record_id'] == record_id] if not sq.empty else pd.DataFrame()
    de_rec = de[de['record_id'] == record_id] if not de.empty else pd.DataFrame()
    tw_rec = tw[tw['record_id'] == record_id] if not tw.empty else pd.DataFrame()
    mr_rec = mr[mr['record_id'] == record_id] if not mr.empty else pd.DataFrame()

    failure_type = 'arrhythmia'
    score = 0.2
    confidence_impact = 0.8

    if not sq_rec.empty and sq_rec['signal_quality_score'].mean() < 0.5:
        failure_type = 'bw'
        score = float(1.0 - sq_rec['signal_quality_score'].mean())
        confidence_impact = score
    elif not sq_rec.empty and sq_rec['hfn_index'].mean() > 0.15:
        failure_type = 'hfn'
        score = float(sq_rec['hfn_index'].mean())
        confidence_impact = 0.6
    elif not sq_rec.empty and sq_rec['pli_index'].mean() > 0.03:
        failure_type = 'pli'
        score = float(sq_rec['pli_index'].mean())
        confidence_impact = 0.6
    elif not sq_rec.empty and 'electrode_motion_index' in sq_rec.columns and sq_rec['electrode_motion_index'].mean() > 0.4:
        failure_type = 'em'
        score = float(sq_rec['electrode_motion_index'].mean())
        confidence_impact = 0.6
    elif not tw_rec.empty and 't_end_ambiguity_score' in tw_rec.columns and tw_rec['t_end_ambiguity_score'].mean() > 0.4:
        failure_type = 'twave_ambiguity'
        score = float(tw_rec['t_end_ambiguity_score'].mean())
        confidence_impact = float(1.0 - tw_rec['t_end_ambiguity_score'].mean())
    elif not mr_rec.empty and mr_rec['lead_agreement_score'].mean() < 0.7:
        failure_type = 'lead_disagreement'
        score = float(1.0 - mr_rec['lead_agreement_score'].mean())
        confidence_impact = score
    elif not de_rec.empty and de_rec['boundary_confidence'].mean() < 0.7:
        failure_type = 'delineation_failure'
        score = float(1.0 - de_rec['boundary_confidence'].mean())
        confidence_impact = score

    rows.append({
        'record_id': record_id,
        'failure_type': failure_type,
        'failure_score': score,
        'confidence_impact': confidence_impact,
        'root_cause_rank': 1,
    })

failure_df = pd.DataFrame(rows)
expected = {'record_id','failure_type','failure_score','confidence_impact','root_cause_rank'}
assert expected.issubset(set(failure_df.columns)), 'failure modes schema mismatch'
assert failure_df['record_id'].is_unique

failure_df.to_parquet(artifacts_dir / 'failure_modes.parquet', index=False)
print('Wrote failure_modes.parquet')
